In [ ]:


import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import sqlite3
from pathlib import Path

In [ ]:
# making SQL tables of the csv files
csv_subscription = "subscriptions.csv"
csv_customers = 'customers.csv'
csv_revenue = 'revenue.csv'

# ✅ 2) Create/connect to a SQLite database file
db_path = "saas.db"
conn = sqlite3.connect(db_path)

# ✅ 3) Read CSV
subscriptions = pd.read_csv(csv_subscription)
customers = pd.read_csv(csv_customers)
revenue = pd.read_csv(csv_revenue)

# ✅ 4) Write to SQL table
subscriptions.to_sql("subscriptions", conn, if_exists="replace", index=False)
customers.to_sql("customers", conn, if_exists="replace", index=False)
revenue.to_sql("revenue", conn, if_exists="replace", index=False)

# conn.close()

988

In [ ]:
# list of subscribers
unique_subscribers = """SELECT DISTINCT customer_id FROM subscriptions"""
df_unique_subscribers = pd.read_sql(unique_subscribers, conn)
df_unique_subscribers.shape[0]

168

In [ ]:
# unique list of months
unique_months = """SELECT DISTINCT month
                FROM subscriptions
                ORDER BY month ASC"""
df_unique_months = pd.read_sql(unique_months, conn)
df_unique_months.shape[0]

18

In [ ]:
subscriber_table = """
SELECT s.customer_id, m.month
FROM (
  SELECT DISTINCT customer_id
  FROM subscriptions
) s
CROSS JOIN (
  SELECT DISTINCT month
  FROM subscriptions
) m;"""

df_subscriber = pd.read_sql(subscriber_table, conn)
df_subscriber

,customer_id,month
0,1020,2024-10
1,1020,2024-11
2,1020,2024-12
3,1020,2025-01
4,1020,2025-02
...,...,...
3019,1999,2024-08
3020,1999,2024-09
3021,1999,2024-01
3022,1999,2024-02


In [ ]:
# confirm shape
print(df_subscriber.shape)

#Each customer appears 18 times
print(df_subscriber['customer_id'].value_counts().describe())

# Each month appears 168 times
print(df_subscriber['month'].value_counts().describe())

# Confirm uniqueness of the grid
print(df_subscriber.duplicated(subset=["customer_id", "month"]).sum())



(3024, 2)
count    168.0
mean      18.0
std        0.0
min       18.0
25%       18.0
50%       18.0
75%       18.0
max       18.0
Name: count, dtype: float64
count     18.0
mean     168.0
std        0.0
min      168.0
25%      168.0
50%      168.0
75%      168.0
max      168.0
Name: count, dtype: float64
0


In [ ]:
# 1) Create df_activity (customer-month unique)
df_activity = subscriptions[['customer_id', 'month']].drop_duplicates()
df_activity.duplicated(['customer_id', 'month']).sum()
df_activity.shape

(988, 2)

In [ ]:
# 2) Left-join activity onto the full grid
df_panel = df_subscriber.merge(df_activity, on=['customer_id', 'month'], how='left', indicator=True)
df_panel

,customer_id,month,_merge
0,1020,2024-10,both
1,1020,2024-11,both
2,1020,2024-12,both
3,1020,2025-01,both
4,1020,2025-02,both
...,...,...,...
3019,1999,2024-08,left_only
3020,1999,2024-09,left_only
3021,1999,2024-01,left_only
3022,1999,2024-02,left_only


In [ ]:
# 3) Create is_active
df_panel['is_active'] = (df_panel['_merge'] == "both").astype(int)
df_panel = df_panel.drop(columns =['_merge'])
df_panel

,customer_id,month,is_active
0,1020,2024-10,1
1,1020,2024-11,1
2,1020,2024-12,1
3,1020,2025-01,1
4,1020,2025-02,1
...,...,...,...
3019,1999,2024-08,0
3020,1999,2024-09,0
3021,1999,2024-01,0
3022,1999,2024-02,0


In [ ]:
print(df_panel.shape)
print(df_panel["is_active"].value_counts())

panel_active = df_panel.groupby("month")["is_active"].sum()
print(panel_active)

(3024, 3)
is_active
0    2036
1     988
Name: count, dtype: int64
month
2024-01    10
2024-02    17
2024-03    30
2024-04    39
2024-05    49
2024-06    53
2024-07    60
2024-08    65
2024-09    64
2024-10    72
2024-11    70
2024-12    74
2025-01    80
2025-02    87
2025-03    87
2025-04    66
2025-05    44
2025-06    21
Name: is_active, dtype: int64


In [ ]:
subs_active = df_activity.groupby("month")["customer_id"].nunique()

(panel_active == subs_active).all()


np.True_

In [ ]:
df_active_monthly = (
    df_panel.groupby('month', as_index=False)['is_active']
    .sum()
    .rename(columns={'is_active': 'active_subscribers'})
)

df_active_monthly

,month,active_subscribers
0,2024-01,10
1,2024-02,17
2,2024-03,30
3,2024-04,39
4,2024-05,49
5,2024-06,53
6,2024-07,60
7,2024-08,65
8,2024-09,64
9,2024-10,72


In [ ]:
df_active_monthly.shape
print(df_active_monthly["month"].nunique())


18


**Cross-check vs subscriptions (source of truth)**

In [ ]:
subs_active = subscriptions.groupby("month")["customer_id"].nunique().reset_index(name="active_subscribers_truth")

check = df_active_monthly.merge(subs_active, on='month', how='inner')

(check['active_subscribers'] == check['active_subscribers_truth']).all()

np.True_

# New Subscribers (Monthly)

**1. Derive first active month per customer**

In [ ]:
df_first_active_month = (
    subscriptions
    .groupby('customer_id')['month']
    .min()
    .reset_index()
    .rename(columns={"month": "first_active_month"})
)
df_first_active_month


,customer_id,first_active_month
0,1020,2024-10
1,1021,2024-04
2,1023,2024-12
3,1026,2025-03
4,1031,2024-04
...,...,...
163,1975,2024-02
164,1982,2024-06
165,1986,2024-06
166,1988,2024-11


**2. Count by month**

In [ ]:
df_new_subscribers = (
    df_first_active_month.groupby('first_active_month')['customer_id']
    .count()
    .reset_index()
    .rename(columns={"customer_id": "new_subscribers",
                     "first_active_month": "month"})
)
df_new_subscribers

,month,new_subscribers
0,2024-01,10
1,2024-02,8
2,2024-03,13
3,2024-04,10
4,2024-05,10
5,2024-06,10
6,2024-07,10
7,2024-08,8
8,2024-09,6
9,2024-10,11


# Churned subscribers per month

In [ ]:
df_churn = customers[customers["churn_date"].notna()]

In [ ]:
df_churn["churn_date"].dtype
# since it is not in datetime first convert
dt_churn_date = pd.to_datetime(df_churn["churn_date"], errors='coerce')

# extract month from date
df_churn['churn_month'] = dt_churn_date.dt.to_period('M')

df_churned_monthly = (
    df_churn.groupby('churn_month')['customer_id']
    .nunique()
    .reset_index()
    .rename(columns={"customer_id": "churned_subscribers",})
)

df_churned_monthly

/tmp/ipykernel_410/2931053075.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_churn['churn_month'] = dt_churn_date.dt.to_period('M')


,churn_month,churned_subscribers
0,2024-02,1
1,2024-04,1
2,2024-05,2
3,2024-06,5
4,2024-07,4
5,2024-08,4
6,2024-09,6
7,2024-10,6
8,2024-11,8
9,2024-12,6


# Churn Rate (Monthly)

In [ ]:
# convert the values in df_churned_monthly column to string

df_churned_monthly['month'] = df_churned_monthly['churn_month'].astype(str)
df_churned_monthly = df_churned_monthly.drop(columns=['churn_month'])

# Combine activity + churn
df_activity_churn_merge = pd.merge(df_active_monthly, df_churned_monthly, on='month', how='left')

# convert NaN into 0
df_activity_churn_merge['churned_subscribers'] = df_activity_churn_merge["churned_subscribers"].fillna(0)

# Create previous-month active
df_activity_churn_merge = df_activity_churn_merge.sort_values('month')
df_activity_churn_merge['prev_active_subscribers'] = df_activity_churn_merge['active_subscribers'].shift(1)
df_activity_churn_merge['churn_rate_monthly'] = df_activity_churn_merge['churned_subscribers'] / df_activity_churn_merge['prev_active_subscribers']

df_activity_churn_merge

,month,active_subscribers,churned_subscribers,prev_active_subscribers,churn_rate_monthly
0,2024-01,10,0.0,NaN,NaN
1,2024-02,17,1.0,10.0,0.100000
2,2024-03,30,0.0,17.0,0.000000
3,2024-04,39,1.0,30.0,0.033333
4,2024-05,49,2.0,39.0,0.051282
5,2024-06,53,5.0,49.0,0.102041
6,2024-07,60,4.0,53.0,0.075472
7,2024-08,65,4.0,60.0,0.066667
8,2024-09,64,6.0,65.0,0.092308
9,2024-10,72,6.0,64.0,0.093750


# MRR(Monthly recurring revenue)

In [ ]:
mrr = (
    revenue.groupby('month')['amount']
    .sum()
    .reset_index()
    .rename(columns={"amount": "MRR"})
)

mrr

,month,MRR
0,2024-01,1850
1,2024-02,4600
2,2024-03,7650
3,2024-04,9600
4,2024-05,11750
5,2024-06,12700
6,2024-07,15900
7,2024-08,17800
8,2024-09,17150
9,2024-10,18750


# ARR(Anual recurring revenue)

In [ ]:
mrr['ARR'] = mrr['MRR'] * 12
mrr

,month,MRR,ARR
0,2024-01,1850,22200
1,2024-02,4600,55200
2,2024-03,7650,91800
3,2024-04,9600,115200
4,2024-05,11750,141000
5,2024-06,12700,152400
6,2024-07,15900,190800
7,2024-08,17800,213600
8,2024-09,17150,205800
9,2024-10,18750,225000


In [ ]:
df_activity_churn_merge['MRR'] = mrr['MRR']
df_activity_churn_merge['ARR'] = mrr['ARR']

# Retention rate

In [ ]:
df_activity_churn_merge['retention_rate'] = 1 - df_activity_churn_merge['churn_rate_monthly']
df_activity_churn_merge

,month,active_subscribers,churned_subscribers,prev_active_subscribers,churn_rate_monthly,MRR,ARR,retention_rate
0,2024-01,10,0.0,NaN,NaN,1850,22200,NaN
1,2024-02,17,1.0,10.0,0.100000,4600,55200,0.900000
2,2024-03,30,0.0,17.0,0.000000,7650,91800,1.000000
3,2024-04,39,1.0,30.0,0.033333,9600,115200,0.966667
4,2024-05,49,2.0,39.0,0.051282,11750,141000,0.948718
5,2024-06,53,5.0,49.0,0.102041,12700,152400,0.897959
6,2024-07,60,4.0,53.0,0.075472,15900,190800,0.924528
7,2024-08,65,4.0,60.0,0.066667,17800,213600,0.933333
8,2024-09,64,6.0,65.0,0.092308,17150,205800,0.907692
9,2024-10,72,6.0,64.0,0.093750,18750,225000,0.906250


# Customer Lifetime

In [ ]:
average_churn_rate = df_activity_churn_merge['churn_rate_monthly'].mean()
customer_lifetime_months = 1 / average_churn_rate
customer_lifetime_months

np.float64(6.0470671901578195)

In [ ]:
median_churn_rate = df_activity_churn_merge['churn_rate_monthly'].median()
customer_lifetime_months = 1 / median_churn_rate
customer_lifetime_months

10.833333333333332

Using the median monthly churn rate to avoid distortion from late-stage churn spikes, the estimated customer lifetime is ~10.8 months.

This step answers the money question:

“Is this SaaS economically viable?”

More precisely:

Are customers worth more than they cost to acquire?

If not → growth is meaningless, even if MRR grows

# ARPU(Average revenue per user)

In [ ]:
df_activity_churn_merge['ARPU'] = df_activity_churn_merge['MRR'] / df_activity_churn_merge['active_subscribers']
df_activity_churn_merge

,month,active_subscribers,churned_subscribers,prev_active_subscribers,churn_rate_monthly,MRR,ARR,retention_rate,ARPU
0,2024-01,10,0.0,NaN,NaN,1850,22200,NaN,185.000000
1,2024-02,17,1.0,10.0,0.100000,4600,55200,0.900000,270.588235
2,2024-03,30,0.0,17.0,0.000000,7650,91800,1.000000,255.000000
3,2024-04,39,1.0,30.0,0.033333,9600,115200,0.966667,246.153846
4,2024-05,49,2.0,39.0,0.051282,11750,141000,0.948718,239.795918
5,2024-06,53,5.0,49.0,0.102041,12700,152400,0.897959,239.622642
6,2024-07,60,4.0,53.0,0.075472,15900,190800,0.924528,265.000000
7,2024-08,65,4.0,60.0,0.066667,17800,213600,0.933333,273.846154
8,2024-09,64,6.0,65.0,0.092308,17150,205800,0.907692,267.968750
9,2024-10,72,6.0,64.0,0.093750,18750,225000,0.906250,260.416667


In [ ]:
# merging new_subcriber with df_activity_churn_merge(main table)

df_activity_churn_merge = df_activity_churn_merge.merge(
    df_new_subscribers[['month', 'new_subscribers']],
    on='month',
    how='left'
)
df_activity_churn_merge

,month,active_subscribers,churned_subscribers,prev_active_subscribers,churn_rate_monthly,MRR,ARR,retention_rate,ARPU,new_subscribers
0,2024-01,10,0.0,NaN,NaN,1850,22200,NaN,185.000000,10
1,2024-02,17,1.0,10.0,0.100000,4600,55200,0.900000,270.588235,8
2,2024-03,30,0.0,17.0,0.000000,7650,91800,1.000000,255.000000,13
3,2024-04,39,1.0,30.0,0.033333,9600,115200,0.966667,246.153846,10
4,2024-05,49,2.0,39.0,0.051282,11750,141000,0.948718,239.795918,10
5,2024-06,53,5.0,49.0,0.102041,12700,152400,0.897959,239.622642,10
6,2024-07,60,4.0,53.0,0.075472,15900,190800,0.924528,265.000000,10
7,2024-08,65,4.0,60.0,0.066667,17800,213600,0.933333,273.846154,8
8,2024-09,64,6.0,65.0,0.092308,17150,205800,0.907692,267.968750,6
9,2024-10,72,6.0,64.0,0.093750,18750,225000,0.906250,260.416667,11


**Create a proper month-start date**

In [ ]:
df_activity_churn_merge['month_start'] = pd.to_datetime(
    df_activity_churn_merge['month'] + '-01'
)

**Create Net Subscriber Change**

In [ ]:
df_activity_churn_merge['net_subscriber_change'] = (
    df_activity_churn_merge['new_subscribers'] -
    df_activity_churn_merge['churned_subscribers']
)
df_activity_churn_merge

,month,active_subscribers,churned_subscribers,prev_active_subscribers,churn_rate_monthly,MRR,ARR,retention_rate,ARPU,new_subscribers,month_start,net_subscriber_change
0,2024-01,10,0.0,NaN,NaN,1850,22200,NaN,185.000000,10,2024-01-01,10.0
1,2024-02,17,1.0,10.0,0.100000,4600,55200,0.900000,270.588235,8,2024-02-01,7.0
2,2024-03,30,0.0,17.0,0.000000,7650,91800,1.000000,255.000000,13,2024-03-01,13.0
3,2024-04,39,1.0,30.0,0.033333,9600,115200,0.966667,246.153846,10,2024-04-01,9.0
4,2024-05,49,2.0,39.0,0.051282,11750,141000,0.948718,239.795918,10,2024-05-01,8.0
5,2024-06,53,5.0,49.0,0.102041,12700,152400,0.897959,239.622642,10,2024-06-01,5.0
6,2024-07,60,4.0,53.0,0.075472,15900,190800,0.924528,265.000000,10,2024-07-01,6.0
7,2024-08,65,4.0,60.0,0.066667,17800,213600,0.933333,273.846154,8,2024-08-01,4.0
8,2024-09,64,6.0,65.0,0.092308,17150,205800,0.907692,267.968750,6,2024-09-01,0.0
9,2024-10,72,6.0,64.0,0.093750,18750,225000,0.906250,260.416667,11,2024-10-01,5.0


Exporting this table for Power BI

In [ ]:
df_activity_churn_merge.to_csv("monthly_metrics.csv", index=False)

In [ ]:
# median ARPU
median_arpu = df_activity_churn_merge['ARPU'].median()

median_arpu

255.35714285714286

# Life Time Value(LTV)

In [ ]:
LTV = median_arpu * customer_lifetime_months
LTV

2766.3690476190473

# Customer acquisition cost (CAC)

In [ ]:
total_acquisition_cost = customers['acquisition_cost'].sum()


In [ ]:
CAC = total_acquisition_cost / df_unique_subscribers.shape[0]
CAC


np.float64(655.4166666666666)

# LTV / CAC ratio

In [ ]:
LTV_CAC_ratio = LTV / CAC
LTV_CAC_ratio

np.float64(4.22077922077922)

# Retention Cohort

In [ ]:

customers['signup_date'] = pd.to_datetime(customers['signup_date'])

customers['cohort_month'] = customers['signup_date'].dt.to_period('M')

customers['cohort_month']

,cohort_month
0,2024-11
1,2024-06
2,2024-12
3,2024-11
4,2024-08
...,...
995,2025-02
996,2025-02
997,2024-01
998,2025-04


In [ ]:
subscriptions

,subscription_id,customer_id,month,monthly_fee
0,S-1020-202410,1020,2024-10,200
1,S-1020-202411,1020,2024-11,200
2,S-1020-202412,1020,2024-12,200
3,S-1020-202501,1020,2025-01,200
4,S-1020-202502,1020,2025-02,200
...,...,...,...,...
983,S-1988-202501,1988,2025-01,500
984,S-1988-202502,1988,2025-02,500
985,S-1988-202503,1988,2025-03,500
986,S-1999-202504,1999,2025-04,500


Merge with subscription

In [ ]:
subscription_with_cohort = subscriptions.merge(
    customers[['customer_id', 'cohort_month']],
    on='customer_id',
    how='left'
)

subscription_with_cohort

,subscription_id,customer_id,month,monthly_fee,cohort_month
0,S-1020-202410,1020,2024-10,200,2024-10
1,S-1020-202411,1020,2024-11,200,2024-10
2,S-1020-202412,1020,2024-12,200,2024-10
3,S-1020-202501,1020,2025-01,200,2024-10
4,S-1020-202502,1020,2025-02,200,2024-10
...,...,...,...,...,...
983,S-1988-202501,1988,2025-01,500,2024-11
984,S-1988-202502,1988,2025-02,500,2024-11
985,S-1988-202503,1988,2025-03,500,2024-11
986,S-1999-202504,1999,2025-04,500,2025-04


In [ ]:
cohort_counts = (
    subscription_with_cohort
    .groupby(['cohort_month', 'month'])['customer_id']
    .nunique()
    .reset_index(name='active_customers')
    )

cohort_counts

,cohort_month,month,active_customers
0,2024-01,2024-01,10
1,2024-01,2024-02,9
2,2024-01,2024-03,9
3,2024-01,2024-04,9
4,2024-01,2024-05,9
...,...,...,...
152,2025-04,2025-04,8
153,2025-04,2025-05,8
154,2025-04,2025-06,1
155,2025-05,2025-05,4


In [ ]:
(cohort_counts["month"] < cohort_counts["cohort_month"]).sum()


np.int64(0)

In [ ]:
# Step 1 — Build a true cohort_size table
cohort_size = (
    subscription_with_cohort
    .groupby("cohort_month")["customer_id"]
    .nunique()
    .reset_index(name="cohort_size")
)

# Step 2 — Merge cohort_size into cohort_counts
cohort_counts2 = cohort_counts.merge(cohort_size, on='cohort_month', how='left')

# Step 3 — Calculate retention
cohort_counts2["retention_rate"] = cohort_counts2["active_customers"] / cohort_counts2["cohort_size"]
cohort_counts2["retention_pct"] = cohort_counts2["retention_rate"] * 100

In [ ]:
cohort_counts2

,cohort_month,month,active_customers,cohort_size,retention_rate,retention_pct
0,2024-01,2024-01,10,10,1.000,100.0
1,2024-01,2024-02,9,10,0.900,90.0
2,2024-01,2024-03,9,10,0.900,90.0
3,2024-01,2024-04,9,10,0.900,90.0
4,2024-01,2024-05,9,10,0.900,90.0
...,...,...,...,...,...,...
152,2025-04,2025-04,8,8,1.000,100.0
153,2025-04,2025-05,8,8,1.000,100.0
154,2025-04,2025-06,1,8,0.125,12.5
155,2025-05,2025-05,4,4,1.000,100.0


# Step 1 — Compute Cohort Index

In [ ]:
cohort_counts2["month"] = pd.to_datetime(cohort_counts2["month"]).dt.to_period('M')

In [ ]:
cohort_counts2["cohort_index"] = (
    cohort_counts2["month"] - cohort_counts2["cohort_month"]
).apply(lambda x: x.n)

# Step 2 — Pivot into Matrix

In [ ]:
cohort_matrix = cohort_counts2.pivot_table(
    index="cohort_month",
    columns="cohort_index",
    values="retention_rate"
)

In [ ]:
cohort_matrix_pct = cohort_matrix * 100
cohort_matrix_pct.round(1)

cohort_index,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
cohort_month,,,,,,,,,,,,,,,,,
2024-01,100.0,90.0,90.0,90.0,90.0,70.0,70.0,70.0,70.0,70.0,60.0,50.0,40.0,30.0,20.0,10.0,10.0
2024-02,100.0,100.0,87.5,87.5,87.5,75.0,75.0,75.0,75.0,62.5,62.5,62.5,62.5,50.0,12.5,12.5,NaN
2024-03,100.0,100.0,100.0,84.6,69.2,69.2,69.2,61.5,53.8,46.2,46.2,46.2,46.2,46.2,38.5,15.4,NaN
2024-04,100.0,100.0,80.0,80.0,70.0,70.0,70.0,60.0,60.0,50.0,40.0,40.0,20.0,NaN,NaN,NaN,NaN
2024-05,100.0,100.0,100.0,80.0,60.0,60.0,50.0,50.0,40.0,40.0,40.0,30.0,20.0,NaN,NaN,NaN,NaN
2024-06,100.0,100.0,100.0,90.0,90.0,70.0,50.0,50.0,50.0,30.0,10.0,10.0,NaN,NaN,NaN,NaN,NaN
2024-07,100.0,100.0,60.0,50.0,50.0,50.0,50.0,50.0,40.0,20.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024-08,100.0,100.0,87.5,75.0,62.5,50.0,50.0,25.0,12.5,12.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024-09,100.0,100.0,83.3,50.0,50.0,50.0,33.3,16.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Converting them to dates for power bi sorting

In [ ]:
cohort_counts2["cohort_month_start"] = cohort_counts2["cohort_month"].dt.to_timestamp()
cohort_counts2["month_start"] = cohort_counts2["month"].dt.to_timestamp()

cohort_counts2

,cohort_month,month,active_customers,cohort_size,retention_rate,retention_pct,cohort_index,cohort_month_start,month_start
0,2024-01,2024-01,10,10,1.000,100.0,0,2024-01-01,2024-01-01
1,2024-01,2024-02,9,10,0.900,90.0,1,2024-01-01,2024-02-01
2,2024-01,2024-03,9,10,0.900,90.0,2,2024-01-01,2024-03-01
3,2024-01,2024-04,9,10,0.900,90.0,3,2024-01-01,2024-04-01
4,2024-01,2024-05,9,10,0.900,90.0,4,2024-01-01,2024-05-01
...,...,...,...,...,...,...,...,...,...
152,2025-04,2025-04,8,8,1.000,100.0,0,2025-04-01,2025-04-01
153,2025-04,2025-05,8,8,1.000,100.0,1,2025-04-01,2025-05-01
154,2025-04,2025-06,1,8,0.125,12.5,2,2025-04-01,2025-06-01
155,2025-05,2025-05,4,4,1.000,100.0,0,2025-05-01,2025-05-01


In [ ]:
cohort_counts2.to_csv("cohort_retention_long.csv", index=False)